# Feature Engineering: 배터리 수명 예측 피처셋 구축

EDA 단계에서 도출된 핵심 인사이트 및 모델 설계 전략을 바탕으로 데이터 누수(Data Leakage) 없는 피처 가공, 데이터 분할, 표준화 파이프라인 수행

## 0. 필수 라이브러리 및 경로 설정

In [2]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler

# 경로 설정
CURRENT_DIR = Path.cwd().resolve()
ROOT = CURRENT_DIR.parent if CURRENT_DIR.name in ('notebooks', 'test') else CURRENT_DIR
INPUT_DIR = ROOT / 'results' / 'eda' / 'tables'
OUTPUT_DIR = ROOT / 'results' / 'features'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'입력 데이터 경로: {INPUT_DIR}')
print(f'피처 출력 경로: {OUTPUT_DIR}')


입력 데이터 경로: /Users/yujin.je/Workspace/ds-mini/results/eda/tables
피처 출력 경로: /Users/yujin.je/Workspace/ds-mini/results/features


## 1. 데이터 로드 및 유효 표본 필터링
- 타겟 결측 셀(10개: Batch 2 8개, Batch 3 2개)은 EOL 미도달 또는 센서 결측으로 지도학습 대상에서 제외
- 결측 타겟의 인위적 대치 없이 유효 셀 129개(Batch 1 46개, Batch 2 39개, Batch 3 44개)만 추출

In [3]:
# EDA 집계 피처 로드
df = pd.read_csv(INPUT_DIR / 'all_batch_cell_features.csv')

# 타겟 결측치 제거 및 유효 표본 필터링
valid_df = df[df.cycle_life.notna()].copy().reset_index(drop=True)

print(f'전체 셀 수: {len(df)}개 | 유효 셀 수: {len(valid_df)}개 (결측 {len(df) - len(valid_df)}개 제외)')
display(valid_df.groupby('batch').agg(
    total=('cell_id', 'count'),
    median_life=('cycle_life', 'median'),
    min_life=('cycle_life', 'min'),
    max_life=('cycle_life', 'max')
))

전체 셀 수: 139개 | 유효 셀 수: 129개 (결측 10개 제외)


,total,median_life,min_life,max_life
batch,,,,
Batch 1,46,858.5,534.0,1227.0
Batch 2,39,472.0,392.0,1186.0
Batch 3,44,1005.5,541.0,1935.0


## 2. 피처 생성 및 선택 (Feature Selection)
- **핵심 피처 (`log_delta_q_var`)**: 3개 배치 전반에서 가장 강력한 음의 상관(-0.7 이하)을 보인 $\Delta Q(V)$ 분산에 $\log_{10}$ 변환을 적용하여 극단적 왜도 완화
- **충전 조건 분해 (`first_C_rate`, `switch_SOC`, `second_C_rate`)**: 원문 범주형의 과적합(소표본/다범주)을 방지하고 일반화 성능을 확보하기 위해 2단계 급속 충전 파라미터를 연속형 수치로 분해
- **물리 보조 피처 (`mean_IR`, `mean_chargetime`, `mean_Tavg`, `mean_QD`, `std_QD`)**: 초기 100 사이클의 내부저항, 충전 소요 시간, 평균 온도, 방전용량 안정성 지표 추가
- **제외 피처**: 미래 정보 누수인 `knee_cycle` 및 사이클 100 이후 통계 완전 배제, 다중공선성(>0.95) 변수인 `delta_q_min`, `mean_Tmax` 제외, Batch 1 내 분산이 없는 `newstructure` 제외

In [4]:
# 1) 핵심 피처 log10 변환
valid_df['log_delta_q_var'] = np.log10(valid_df['delta_q_var'])

# 2) 최종 선정 피처 목록 정의
FEATURE_COLS = [
    'log_delta_q_var',   # 핵심: ΔQ(V) 분산 log10
    'first_C_rate',      # 충전조건: 1차 C-rate
    'switch_SOC',        # 충전조건: 전환 SOC (%)
    'second_C_rate',     # 충전조건: 2차 C-rate
    'mean_IR',           # 보조: 초기 내부저항 평균
    'mean_chargetime',   # 보조: 초기 충전시간 평균
    'mean_Tavg',         # 보조: 초기 평균 온도
    'mean_QD',           # 보조: 초기 방전용량 평균
    'std_QD',            # 보조: 초기 방전용량 표준편차
]
META_COLS = ['batch', 'cell_id', 'charging_policy', 'cycle_life']

print(f'최종 선정 피처 수: {len(FEATURE_COLS)}개')
display(valid_df[FEATURE_COLS].describe().round(4))

최종 선정 피처 수: 9개


,log_delta_q_var,first_C_rate,switch_SOC,second_C_rate,mean_IR,mean_chargetime,mean_Tavg,mean_QD,std_QD
count,129.0000,129.0000,129.0000,129.0000,129.0000,129.0000,129.0000,129.0000,129.0000
mean,-3.9169,5.4039,49.5194,4.1554,0.0153,13.9149,33.2378,1.0805,0.0038
std,0.4277,0.9225,20.8507,0.7423,0.0035,11.4976,1.3151,0.0147,0.0172
min,-5.0989,3.6000,9.0000,3.0000,0.0000,8.9896,29.8922,1.0438,0.0002
25%,-4.1789,4.8000,31.0000,3.6000,0.0153,10.0433,32.2512,1.0703,0.0005
50%,-3.9997,5.4000,50.0000,4.0000,0.0161,10.1736,33.1315,1.0811,0.0008
75%,-3.5163,5.6000,67.0000,4.6000,0.0167,11.1504,33.8977,1.0927,0.0024
max,-3.0859,8.0000,80.0000,6.0000,0.0183,53.3672,36.8377,1.1113,0.1893


## 3. 누수 없는 데이터셋 분할 (Data Splitting)
- **Train / Hold-out Valid 분할 (Batch 1)**: 동일 충전 프로토콜 셀이 Train과 Valid에 동시 투입되는 정보 누수를 차단하기 위해 `charging_policy` 기준 `GroupShuffleSplit` 적용 (Train 35개, Valid 11개)
- **Test 1 (Batch 2)**: 단수명 중심 환경(중앙값 472 cycle)에서의 필수 외부 일반화 검증 데이터셋
- **Test 2 (Batch 3)**: 장수명 중심 환경(중앙값 1,005.5 cycle)에서의 추가 외부 일반화 검증 데이터셋

In [5]:
# Batch 분리
b1 = valid_df[valid_df.batch == 'Batch 1'].copy().reset_index(drop=True)
test_b2 = valid_df[valid_df.batch == 'Batch 2'].copy().reset_index(drop=True)
test_b3 = valid_df[valid_df.batch == 'Batch 3'].copy().reset_index(drop=True)

# Batch 1 내부 GroupShuffleSplit (charging_policy 기준)
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(b1, groups=b1['charging_policy']))

train_df = b1.iloc[train_idx].copy().reset_index(drop=True)
val_df = b1.iloc[val_idx].copy().reset_index(drop=True)

# 분할 결과 검증
policy_overlap = set(train_df.charging_policy).intersection(set(val_df.charging_policy))
assert len(policy_overlap) == 0, 'Train과 Valid 간 정책 중복 발생'

print(f'Train 세트 (Batch 1): {len(train_df)}개 셀 ({train_df.charging_policy.nunique()}개 정책) | 중앙 수명: {train_df.cycle_life.median():.1f}')
print(f'Valid 세트 (Batch 1): {len(val_df)}개 셀 ({val_df.charging_policy.nunique()}개 정책) | 중앙 수명: {val_df.cycle_life.median():.1f}')
print(f'Test 1 세트 (Batch 2): {len(test_b2)}개 셀 ({test_b2.charging_policy.nunique()}개 정책) | 중앙 수명: {test_b2.cycle_life.median():.1f}')
print(f'Test 2 세트 (Batch 3): {len(test_b3)}개 셀 ({test_b3.charging_policy.nunique()}개 정책) | 중앙 수명: {test_b3.cycle_life.median():.1f}')

Train 세트 (Batch 1): 35개 셀 (18개 정책) | 중앙 수명: 857.0
Valid 세트 (Batch 1): 11개 셀 (5개 정책) | 중앙 수명: 860.0
Test 1 세트 (Batch 2): 39개 셀 (12개 정책) | 중앙 수명: 472.0
Test 2 세트 (Batch 3): 44개 셀 (8개 정책) | 중앙 수명: 1005.5


## 4. 누수 없는 표준화 및 최종 데이터셋 저장
- 평가 데이터 정보의 사전 유입(Data Leakage)을 방지하기 위해 `StandardScaler`는 오직 Train 세트 기준으로만 `fit` 수행
- Valid 및 Test 세트는 Train 기준 스케일러로 단순 `transform` 적용
- 선형 규제 모델용 표준화 데이터셋(Scaled)과 트리 계열 모델용 원본 데이터셋(Raw) 2종으로 분리 저장

In [6]:
# 1) Raw 데이터셋 구성
all_cols = META_COLS + FEATURE_COLS
train_raw = train_df[all_cols].copy()
val_raw = val_df[all_cols].copy()
test_b2_raw = test_b2[all_cols].copy()
test_b3_raw = test_b3[all_cols].copy()

# 2) Scaled 데이터셋 구성 (Train 기준 Fit)
scaler = StandardScaler()
train_scaled = train_raw.copy()
val_scaled = val_raw.copy()
test_b2_scaled = test_b2_raw.copy()
test_b3_scaled = test_b3_raw.copy()

train_scaled[FEATURE_COLS] = scaler.fit_transform(train_raw[FEATURE_COLS])
val_scaled[FEATURE_COLS] = scaler.transform(val_raw[FEATURE_COLS])
test_b2_scaled[FEATURE_COLS] = scaler.transform(test_b2_raw[FEATURE_COLS])
test_b3_scaled[FEATURE_COLS] = scaler.transform(test_b3_raw[FEATURE_COLS])

# 3) CSV 저장
datasets = {
    'train_features_raw.csv': train_raw,
    'valid_features_raw.csv': val_raw,
    'test_b2_features_raw.csv': test_b2_raw,
    'test_b3_features_raw.csv': test_b3_raw,
    'train_features_scaled.csv': train_scaled,
    'valid_features_scaled.csv': val_scaled,
    'test_b2_features_scaled.csv': test_b2_scaled,
    'test_b3_features_scaled.csv': test_b3_scaled,
}

for filename, dataset in datasets.items():
    dataset.to_csv(OUTPUT_DIR / filename, index=False)

# 4) 최종 무결성 검증
assert len(train_raw) == 35
assert len(val_raw) == 11
assert len(test_b2_raw) == 39
assert len(test_b3_raw) == 44
assert len(list(OUTPUT_DIR.glob('*.csv'))) == 8
print('데이터셋 저장 및 무결성 검증 완료: Raw 4개, Scaled 4개 총 8개 파일 생성')

데이터셋 저장 및 무결성 검증 완료: Raw 4개, Scaled 4개 총 8개 파일 생성
